# Rossmann Sales — ML Experiment

Notebook trình bày quy trình thực nghiệm theo **5 giai đoạn**, giữ nguyên số liệu thực tế và kết thúc bằng LightGBM làm mô hình dự đoán:
- Giai đoạn 1: chuẩn bị dữ liệu và kiểm tra rò rỉ;
- Giai đoạn 2: huấn luyện/đánh giá LightGBM bằng holdout 42 ngày;
- Giai đoạn 3: phân tích metric và sai số so với baseline;
- Giai đoạn 4: giải thích đặc trưng quan trọng;
- Giai đoạn 5: xác nhận LightGBM và lưu artifact production dùng cho dự đoán.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root != root.parent and not (root / "backend").is_dir():
    root = root.parent

if not (root / "backend").is_dir():
    raise RuntimeError("Không tìm thấy project root (thư mục backend/) trong đường dẫn cha")

sys.path.insert(0, str(root / "backend"))

import pandas as pd
import matplotlib.pyplot as plt

from app.model.train_model import train_and_evaluate
from app.services.data_service import get_stores, get_train

VALIDATION_DAYS = 42
N_ESTIMATORS = 420

print(f"Project root: {root}")

## Giai đoạn 1 — Kiểm tra dữ liệu và phạm vi holdout

Holdout sử dụng 42 ngày cuối. Mọi dữ liệu sau ngày cutoff được giữ riêng khỏi quá trình huấn luyện.

In [ ]:
train = get_train()
stores = get_stores()

overview = pd.DataFrame([{
    "store_count": stores["Store"].nunique(),
    "train_rows": len(train),
    "date_start": train["Date"].min().date(),
    "date_end": train["Date"].max().date(),
    "open_rows": int(train["Open"].eq(1).sum()),
    "positive_sales_rows": int((train["Open"].eq(1) & train["Sales"].gt(0)).sum()),
}])
display(overview)

if train["Date"].isna().any():
    raise ValueError("Dữ liệu train chứa Date bị thiếu")

## Giai đoạn 2 — Huấn luyện và đánh giá holdout 42 ngày

Lần chạy này tạo validation LightGBM, tính baseline theo `(Store, DayOfWeek, Promo)`, rồi huấn luyện lại mô hình production trên toàn bộ lịch sử hợp lệ.

In [ ]:
metrics = train_and_evaluate(
    validation_days=VALIDATION_DAYS,
    n_estimators=N_ESTIMATORS,
)

comparison = pd.DataFrame.from_dict(
    {
        "baseline": metrics["baseline"],
        "lightgbm": metrics["validation"],
    },
    orient="index",
).rename_axis("model").reset_index()

comparison["mae_improvement_pct"] = (
    (comparison.loc[comparison["model"] == "baseline", "mae"].iloc[0] - comparison["mae"])
    / comparison.loc[comparison["model"] == "baseline", "mae"].iloc[0]
    * 100
)
comparison

## Giai đoạn 3 — Phân tích metric và sai số

- `MAE`, `RMSE`, `WAPE`: càng thấp càng tốt.
- `RMSPE`: chỉ tính trên các ngày có doanh số dương.
- Cột `mae_improvement_pct` dương cho biết mô hình giảm MAE so với baseline.

In [ ]:
display(comparison.style.format({
    "mae": "{:.2f}",
    "rmse": "{:.2f}",
    "rmspe": "{:.5f}",
    "wape": "{:.5f}",
    "mae_improvement_pct": "{:+.2f}%",
}))

print("Validation period:", " to ".join(metrics["validation_dates"]))
print("Training period:", " to ".join(metrics["train_dates"]))

In [ ]:
importance = pd.DataFrame(metrics["important_features"])

fig, ax = plt.subplots(figsize=(9, 6))
importance.sort_values("importance").plot.barh(
    x="feature", y="importance", ax=ax, legend=False, color="#4C78A8"
)
ax.set_title("Top LightGBM Feature Importance")
ax.set_xlabel("Split importance")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()

## Giai đoạn 4 — Giải thích đặc trưng LightGBM

Biểu đồ trên cho biết mức đóng góp tương đối của các đặc trưng trong mô hình LightGBM.

## Giai đoạn 5 — Chốt mô hình và artifact dự đoán

Khi chạy thành công, quy trình chốt và lưu:
- `models/rossmann_sales_lgbm.joblib` — mô hình production cuối cùng;
- `results/evaluation_metrics.json` — metrics holdout và metadata.
- `reports/selected_model.json` — tên, tham số và lý do chọn LightGBM.
- `reports/selected_model.joblib` — artifact riêng của LightGBM nếu đã chạy đầy đủ bộ experiments.

LightGBM được giữ làm mô hình cuối vì có MAE cross-validation tốt nhất trong các mô hình được so sánh, đồng thời hợp lý cho quan hệ phi tuyến và được huấn luyện lại trên toàn bộ lịch sử trước khi dự đoán. Không chỉnh các cột `Sales` hoặc `Customers` trong đặc trưng đầu vào; business rule vẫn ép cửa hàng đóng cửa dự đoán bằng 0.